# 25. Revenue by category when a product sits in several

**Track:** sql | **Difficulty:** medium | **Tags:** many-to-many, bridge table, allocation, joins

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("025-revenue-by-category-many-to-many")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "product_categories": """
UEFSMRUEFaADFeoBTBU0FQASAADQAQQBAAkBAA4JBwQADQ0IABgNCAASDQgADA0IABMNCAAaDQgAFw0IABUNCAADDQgABA0IAAgN
CAAGDQgACg0IAAUNCAACDQgACQ0IABYNCAAZDQgAEQ0IABQNCAALDQgADw0IPBAAAAAAAAAABwAAAAAAAAAVABVMFVAsFVYVEBUG
FQYcGAgaAAAAAAAAABgIAQAAAAAAAAAWACgIGgAAAAAAAAAYCAEAAAAAAAAAEREAAAAmlAIAAABWAQUNIIhBijkoqcWaezDFaAiU
4c0X5KN1WElXiRdnAAAAFQQVaBVoTBUKFQASAAA0fAsAAABFbGVjdHJvbmljcwQAAABIb21lBgAAAE9mZmljBQo4VHJhdmVsBQAA
AEdpZnRzFQAVNBU4LBVWFRAVBhUGHDYAKAZUcmF2ZWwYC0VsZWN0cm9uaWNzEREAAAAaZAIAAABWAQMNSIBACpBs08ZlICcAC4SR
0AAAFQQZPDUAGAZzY2hlbWEVBAAVBCUCGApwcm9kdWN0X2lkABUMJQIYCGNhdGVnb3J5JQBMHAAAABZWGRwZLCYAHBUEGTUABhAZ
GApwcm9kdWN0X2lkFQIWVhaKBRbYAyaSAiYIHBgIGgAAAAAAAAAYCAEAAAAAAAAAFgAoCBoAAAAAAAAAGAgBAAAAAAAAABERABks
FQQVABUCABUAFRAVAgA8KQYZJgBWAAAAJgAcFQwZNQAGEBkYCGNhdGVnb3J5FQIWVhaQAhaUAibkBCbgAxw2ACgGVHJhdmVsGAtF
bGVjdHJvbmljcxERABksFQQVABUCABUAFRAVAgA8FtwEGQYZJgBWAAAAFpoHFlYmCBbsBQAZHBgMQVJST1c6c2NoZW1hGIACLy8v
Ly83Z0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQUlBQUFCSUFB
QUFCQUFBQU5ELy8vOEFBQUVGRUFBQUFDQUFBQUFFQUFBQUFBQUFBQWdBQUFCallYUmxaMjl5ZVFBQUFBQUVBQVFBQkFBQUFCQUFG
QUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFBQWtBQUFBQkFBQUFBQUFBQUFLQUFBQWNISnZaSFZqZEY5cFpBQUFDQUFN
QUFnQUJ3QUlBQUFBQUFBQUFVQUFBQUFBQUFBQQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGSwcAAAcAAAAUgIA
AFBBUjE=
""",
    "sales": """
UEFSMRUEFYAZFd4MTBWQAxUAEgAAwAwMgTgBAAEBAIINCACDDQgAhA0IAIUNCACGDQgAhw0IAIgNCACJDQgAig0IAIsNCACMDQgA
jQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACVDQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0I
AKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgAqA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALIN
CACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADF
DQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0NCADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA
2A0IANkNCADaDQgA2w0IANwNCADdDQgA3g0IAN8NCADgDQgA4Q0IAOINCADjDQgA5A0IAOUNCADmDQgA5w0IAOgNCADpDQgA6g0I
AOsNCADsDQgA7Q0IAO4NCADvDQgA8A0IAPENCADyDQgA8w0IAPQNCAD1DQgA9g0IAPcNCAD4DQgA+Q0IAPoNCAD7DQgA/A0IAP0N
CAD+DQgA/w0IBAA5afgAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgA
EA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcNCAAYDQgAGQ0IABoNCAAbDQgAHA0IAB0NCAAeDQgAHw0IACANCAAhDQgAIg0I
ACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAtDQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUN
CAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgAPQ0IAD4NCAA/DQgAQA0IAEENCABCDQgAQw0IAEQNCABFDQgARg0IPEc5AQAA
AAAASDkBAAAAAAAVABWiAxWqAywVkAMVEBUGFQYcGAhIOQEAAAAAABgIgTgBAAAAAAAWACgISDkBAAAAAAAYCIE4AQAAAAAAEREA
AADRAfDQAwAAAJADAQgzAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7
PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWG
h4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxscVBBXgAxWOAkwV
PBUAEgAA8AEEGAAJAQAOCQcEAA0NCAAIDQgAEQ0IAAMNCAAcDQgABQ0IAAYNCAAPDQgACQ0IABYNCAAEDQgEHQAJAQAZCQcEAAsN
CAAKDQgAAQ0IAAcNCAAMDQgAEg0IAB4NCAAUDQgAEA0IABsNCAATDQgAFw0IAAINCDwaAAAAAAAAABUAAAAAAAAAFQAVjAIVlAIs
FZADFRAVBhUGHBgIHgAAAAAAAAAYCAEAAAAAAAAAFgAoCB4AAAAAAAAAGAgBAAAAAAAAABERAAAAhgHwhQMAAACQAwEFMyCIQYo5
KBSRlFmsOfcgik05YiCRk1Z7WRujAOzcuYiE6iRcs6VRIs1R72LKWPLkwSULIa2+AR8yBTUS67igKxuKCup1pcx7Z4KIKcWINqwN
mVSph1jwRcsoOa1NhLUlCzLVwIHyLUNUCft1REMsjeJ8DmxNDuaarYa3CDg3FQQVvA8Vxg9MFYwDFQASAADeB/TdAwAADtecAAAO
0PgAAAlHAAAAA6/AAAAMkMwAAAFnxAAAAvgoAAAP6jgAABE9rAAADyNkAAAJ2RgAAA8gRAAADXTsAAAODzgAAACECAAAEj8YAAAR
RXwAAA2m7AAABuYEAAAQQIwAAAMERAAACxkEAAALkhwAABJHsAAAEjh0AAAMvhwAAASx9AAAAuN0AAAP3CgAABAzqAAAD3PcAAAE
DLgAAAREMAAADXy8AAAAPhwAAAhU0AAAALc0AAAJTaQAAAfqkAAAAa1MAAAOaXQAAAJepAAADGDAAAAB9SwAAA2LlAAAA19IAAAQ
uNwAAA349AAABtqwAAAPZKAAAAWGEAAADbXEAAAReXAAABINGAAACjQcAAAQeZQAAAsuHAAADSq0AAADQNAAAAXYfAAACK5EAAAO
73AAAAX7QAAABbv4AAACAOQAABABRAAABfNwAAABA2AAAAh9DAAADRTUAAAHMKAAAAxdPAAADbVgAAAOgOQAAAwLmAAADA2MAAAK
Z6wAAAKWHAAAELPIAAAJFcgAAAqBdAAAAZUUAAAQkKAAAAKwSAAACksoAAAIthQAAAL10AAABQTEAAANoqAAAAO7eAAABNDQAAAL
yyQAAAIdBAAADfx4AAABfggAAA29+AAAAJi8AAAGIxgAAApjYAAADEMQAAAKQQAAAAGxmAAACbP8AAAJ/QgAAAuaUAAACs2gAAAO
HzwAABIwQAAABBk4AAAHpWwAAAGw0AAADciEAAAIEaAAAAF7TAAADCXEAAANgWwAAAGmRAAAAjsYAAAE36gAAA8azAAAERuwAAAG
G6wAAAM5yAAADnhMAAAOs6wAAAE45AAAAkc0AAAQqaAAAA5byAAACCM0AAAGv1gAABD4iAAACBUkAAAG3jQAAAOAGAAAEF10AAAL
qMQAAAIKqAAAC69oAAAOTbgAAAIPWAAABY6oAAAG29wAAAj0MAAAAfLUAAAGVeAAAAdabAAACy7kAAANzygAAAfKiAAAEJPAAAAA
bJgAAA4X0AAAEM/oAAADlfgAAAMdRAAAAudcAAAHCZAAAA9ZTAAAEissAAAFUVQAABAVlAAABtQMAAAMcSgAAAF/mAAAEBX4AAAH
JUwAAABRQAAAA3+0AAAGxsQAAAtGVAAAEd6cAAAOQAwAAAMGOAAAEYDcAAAD9IAAAAEhEAAACXV8AAAOCogAAAr4/AAAAUYsAAAJ
WDAAAAE5rAAACSxwAAARR3AAABDm9AAACRHgAAAOOcwAAAQXqAAABiVwAAAESnAAAASnaAAAColEAAAEtkAAAAiIYAAACoVcAAAQ
+bQAAAZufBUAFaIDFaoDLBWQAxUQFQYVBhwYBQAAEkewGAUAAAA+HBYAKAUAABJHsBgFAAAAPhwREQAAANEB8NADAAAAkAMBCDMA
AQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpL
TE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWW
l5iZmpucDZ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5urtivL2+v8DBwsPExRUEGUw1ABgGc2NoZW1hFQYAFQQlAhgHc2Fs
ZV9pZAAVBCUCGApwcm9kdWN0X2lkABUOFQoVAhgGYW1vdW50JQoVBBUULFwVBBUUAAAAFpADGRwZPCYAHBUEGTUABhAZGAdzYWxl
X2lkFQIWkAMWyB0WrhEmiA0mCBwYCEg5AQAAAAAAGAiBOAEAAAAAABYAKAhIOQEAAAAAABgIgTgBAAAAAAAREQAZLBUEFQAVAgAV
ABUQFQIAPCkGGSYAkAMAAAAmABwVBBk1AAYQGRgKcHJvZHVjdF9pZBUCFpADFpAHFsYFJuQTJrYRHBgIHgAAAAAAAAAYCAEAAAAA
AAAAFgAoCB4AAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCQAwAAACYAHBUOGTUABhAZGAZhbW91bnQV
AhaQAxbsExb+EybkJib8FhwYBQAAEkewGAUAAAA+HBYAKAUAABJHsBgFAAAAPhwREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAkAMA
AAAWxDgWkAMmCBbyKgAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJB
QU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDSUFBQUFRQUFBQUFRQUFBQ1UvLy8vQUFBQkJ4QUFBQUFnQUFBQUJBQUFB
QUFBQUFBR0FBQUFZVzF2ZFc1MEFBQUlBQXdBQkFBSUFBZ0FBQUFLQUFBQUFnQUFBTXovLy84QUFBRUNFQUFBQUJ3QUFBQUVBQUFB
QUFBQUFBb0FBQUJ3Y205a2RXTjBYMmxrQUFEQS8vLy9BQUFBQVVBQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJ
UUFBQUFJQUFBQUFRQUFBQUFBQUFBQndBQUFITmhiR1ZmYVdRQUNBQU1BQWdBQndBSUFBQUFBQUFBQVVBQUFBQT0AGCBwYXJxdWV0
LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAAC0DAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "025-revenue-by-category-many-to-many"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

product_categories = spark.read.parquet((_data_dir / "product_categories.parquet").resolve().as_posix())
product_categories.createOrReplaceTempView("product_categories")
sales = spark.read.parquet((_data_dir / "sales.parquet").resolve().as_posix())
sales.createOrReplaceTempView("sales")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'category': 'd6733d202028f0762546c234e7ef48c82d41c24c73100984db0877cfe34e826f',
                         'revenue': 'c37ca3d6f0db0f3dc4dcd0162e6e2519485486ea0716e37cacc0136095932edb'},
 'columns': [['category', 'string'], ['revenue', 'double']],
 'fingerprint': 'd3bcedfed804261d8bae5f706802e072dee25e45c969bc44c78a1d7bbe783427',
 'order_matters': False,
 'row_count': 6}

check = make_check(EXPECTED)

## The situation

Finance wants revenue per product category. A product can be listed in more than one
category: a travel adapter is in both Electronics and Travel.

The current report joins sales to categories and sums. Its category totals add up to
far more than the company sold, because a sale of a product in three categories is
counted three times. Finance wants totals that add up, so each sale is split evenly
across the categories its product belongs to.

## Input tables

### `sales`

One row per sale.

| Column | Type | Description |
|---|---|---|
| `sale_id` | BIGINT | Unique per sale. |
| `product_id` | BIGINT | The product sold. |
| `amount` | DECIMAL(10,2) | Sale value in rupees. |

### `product_categories`

Which product is in which category. A product can have several rows, or none. The
table is maintained by hand and a few rows have been entered twice.

| Column | Type | Description |
|---|---|---|
| `product_id` | BIGINT | The product. |
| `category` | STRING | A category the product is listed in. |

## Requirement

Return revenue per category.

- A sale is split evenly across the categories of its product. A sale of 900.00 for a
  product in three categories adds 300.00 to each of the three.
- A row that appears twice in `product_categories` is still one category for that
  product.
- Sales of a product with no category go to a category named `Uncategorized`.
- The revenues of all rows add up to the total of all sales.
- Do not round. Small differences beyond the sixth decimal are ignored by the check.

## Expected output

One row per category, plus `Uncategorized`.

| Column | Type | Description |
|---|---|---|
| `category` | STRING | |
| `revenue` | DOUBLE | The category's share of sales. |

In [ ]:
product_categories.show(5)
sales.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `product_categories`, `sales`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)